# 260813 python 
[First-class Object, scope, closure, lambda, map(), filter(), decorator]

## 1. Today's Goal
- python 개념 복습 및 외부 자료로 실습하기
- 도서관리시스템 CLI 구현 과제 시작하기
- Notion archiving 환경 조성

---

## 2. Key Concepts

## 2.1 변수 스코프
### 변수 스코프
- `local`: 특정 함수 내부 공간에 선언, 
    - closure: 외부에서 계속 참조되는 값 유지 가능

- `global`: 함수 바깥쪽에 선언, 프로그램 파일 전체에서 접근 가능 
    - global keyward로 외부함수에 선언한 변수는, 내부함수에 있는 이름이 같은 변수에 **똑같이 `global`선언을 하지 않는 한** 영향을 주지 않음
    - 내부함수는 global scope에서 접근 불가

**keyword**
- `nonlocal`: 하위 inner function에서 상위 outer function의 local 변수를 수정하겠다고 선언
- `global`: 어느 함수에서든 전역변수를 수정할 때 사용

--> nonlocal과 global을 나눠 쓰는 이유: 
- 코드가 길 경우, 프로그램에 있는 모든 함수가 그 변수를 바꿀 위험이 있음.
- ex) outer에서 선언한 global variable "b" -> inner에서도 global "b" 선언. outer 함수를 참조하는 변수 x, y는 결국에 같은 global variable "b"를 참조하므로, x에서 바꾼 값은 y에서도 바뀌게 됨.
- 모든 함수에서 global을 쓰는 것보다는 오직 변수를 감싸는 outer 함수와 inner 함수 안에서만 변수를 공유하여 안전하게 캡슐화하기 위함.

In [ ]:
# global 선언 예시
x = 10

def outer():
    global x

    x += 10

    print(f"outer function's global x: {x}, id: {id(x)}") 

    def inner():
        global x                            # inner func에서 `global x`` 선언 하지 않으면 inner만의 x일 뿐임
        x += 20

        print(f"local x: {x}, id: {id(x)}") # 숫자가 바뀌면 id도 바뀜

        is_same_object = (x is globals()['x']) 

        print(f"outer global x is inner x? : {is_same_object}")

    inner()


print(f"global x: {x}, id: {id(x)}")

print(outer())

global x: 10, id: 140708844791192
outer function's global x: 20, id: 140708844791512
local x: 40, id: 140708844792152
outer global x is inner x? : True
None


In [54]:
# nonlocal 선언 예시

def outer():
    y = 10

    def inner():
        nonlocal y # 상위 function인 outer()의 y를 수정하겠다고 선언
        y += 10

    inner()
    print(y)

outer()

20


## 2.2. First-Class Function 일급 함수, Higher-order Function 고차 함수
일급 함수
- 함수 = 변수, 함수를 변수처럼 취급
- 함수 -> 정수, 문자열과 동일한 취급 + 변수에 대입 가능 + 다른 함수의 argument(인수)로 전달 가능 + 함수의 결과값으로 반환 가능


고차 함수
- 다음 하나 이상을 만족하는 함수
  - 다른 함수를 argument로 전달받음
  - 연산의 결과로 또 다른 함수를 반환 *반환시 함수는 괄호 빼고

함수 a가 또 다른 함수b를 만들어내고 혹은 반환하는 방식 --> 고도화된 제어. closure / decorator


**closure**
- inner function이 자신이 정의된 enclosing scope의 변수(자유 변수, free variable) 를 기억하고, outer function이 종료된 후에도 그 변수에 접근할 수 있는 구조.
- outer function의 실행이 종료되고 frame이 제거된 이후에도 inner가 참조하는 변수는 closure를 통해 계속 유지됨
- `변수.closure` 를 통해 closure에 연결된 cell(outer scope의 변수를 closure가 계속 참조할 수 있도록 보관) 확인 가능

In [ ]:
def outer(num1):
    def inner(num2):
        return num1 - num2
    
    return inner # inner()를 반환값으로서 내보냄. inner는 first-class function

a = outer(10) 
# outer(num1=10) 실행
# → inner 함수 객체 생성
# → inner가 num1을 참조하므로 num1은 cell로 캡처됨
# → return inner
# → outer()의 실행 Frame은 Call Stack에서 제거됨
# → 하지만 inner가 num1을 계속 참조하므로
#   num1의 값은 Heap영역의 PyCellObject를 통해 유지됨
# → a는 inner 함수 객체를 참조함
print(a(20)) # a(20) = inner(20). 
print(a) # 10 - 20 = -10
b = outer(20) # 
print(b(20))

print(a.__closure__)
print(a.__closure__[0].cell_contents)

# outer의 지역변수 2개 = num1, inner (first-class function)

-10
<function outer.<locals>.inner at 0x109fdff60>
0
(<cell at 0x109fa3e80: int object at 0x103a0aad8>,)
10


## 2.3. lambda, map(), filter()
`lambda parameter, parametere, ...: code`
- 함수를 한줄로 정의
- 일회용 사용 -> 메모리 절약
- 변수 대입하지 않고 일급함수로서 매개변수로 바로 씀
- 변수 담기지 않으면 그냥 소멸
- True/False 계산 가능

`map(function, iterable)`
- iterable에 있는 모든 item에 함수를 적용시킴
- list(map()) 출력

`filter(function, iterable)`
- 함수 조건으로 데이터를 선별함

-> map()과 fliter()에서 iterable은 매개변수로서 function의 argument 역할을 하는 것으로 이해 가능

In [ ]:
# map()
fruits= ['apple', 'banana', 'orange', 'mango' ]
list(map(lambda x: f"**{x}**", fruits))

In [ ]:
# filter()
numbers = [*range(1,11)] # * 사용해서 펼쳐주기
list(filter(lambda x: x % 2 == 1 , numbers))

## 2.4. Decorator
- 고차 함수 기반: 원본 함수(callback)을 인수로 받음 -> 내부에서 가공 -> 새로운 함수 생성 -> 다시 return
- decorator 함수 안에 wrapper()로 함수 실행 준비
- decorator 함수 및 callback 원본 함수 선언한 뒤, 원본 함수 위 `@decorator` 적으면 원본 함수가 알아서 decorator함수의 인수로 작용, 원본함수 실행 시키면 끝

**factorial**
- factorial n!: 1~n까지 모든 양의 정수 곱하기 ex) n * (n-1) * (n-2) * ... * 2 * 1

In [4]:
# time_checker2 = decorator
import time

def time_checker2(callback):
    def wrapper(*args, **kwargs): # 관례적으로 포장된 함수 wrapper로 지음, 핵심 기능에 해당, *args/**kwargs로 정의하여 나중에 넣을 핵심 기능에 어떤 인자가 오더라도 핸들 가능
        start = time.time() # 공통 기능

        result = callback(*args, **kwargs) # 핵심 기능

        end = time.time() # 공통 기능

        print("걸린 시간:", end - start)

        return result

    return wrapper

In [5]:
@time_checker2 ##알아서 time_checker2 인수로 들어감
def factorial2(num):

    if (num < 1):
        return 1
    
    return num * factorial2(num - 1)

factorial2(10)

걸린 시간: 0.0
걸린 시간: 5.793571472167969e-05
걸린 시간: 7.295608520507812e-05
걸린 시간: 8.296966552734375e-05
걸린 시간: 9.417533874511719e-05
걸린 시간: 0.000102996826171875
걸린 시간: 0.00011229515075683594
걸린 시간: 0.00012230873107910156
걸린 시간: 0.00013208389282226562
걸린 시간: 0.0001430511474609375
걸린 시간: 0.00015616416931152344


3628800

---

## 3. Practice

### 실습 코드

In [6]:
# lambda, map(), filter()
base_numbers = [1, 2, 3, 4, 5, 6]

multiplied_numbers = list(map(lambda x: x * 3, base_numbers))

print(multiplied_numbers)

final_filtered_numbers = list(filter(lambda x: x > 10, multiplied_numbers))

print(final_filtered_numbers)

[3, 6, 9, 12, 15, 18]
[12, 15, 18]


In [ ]:
# decorator function

def auto_tracker(callback):
    def wrapper(*arg, **kwargs):
        print(f"원본 함수 <{callback.__name__}> 실행 시작!")

        print(f"[위치 인수: {arg} | 가변 인수: {kwargs}] 입력!")

        result = callback(*arg, **kwargs)

        print(f"<{callback.__name__}> 연산 완료!")
        
        return result

    return wrapper

@auto_tracker
def add_numbers(a, b):
    return a + b

add_numbers(10, b = 20)

---

## 4. Daily Quest

>1번 문항

LEGB 스코프 구조에서 변수 검색 우선순위가 가장 높은 영역과 가장 낮은 영역을 올바르게 짝지은 것은 무엇인가요?
- 답:

>2번 문항

다음 코드가 실행될 때 최종 print(counter)의 출력 결과는 무엇인가요?
- 답:

In [1]:
counter = 0 

def increase():
    global counter
    
    counter += 1 
    
    def local_increase():
        counter = 100 

increase() 
increase() 

print(counter) 

2


>3번 문항

def 키워드 없이 lambda 매개변수: 표현식 형태로 작성하며, 이름이 없는 한 줄짜리 함수를 무엇이라고 부르나요?
- 답: 

>4번 문항

numbers = [1, 2, 3, 4, 5]에 대해 다음 코드를 실행했을 때 result의 값은 무엇인가요? 
- 답:

In [ ]:
numbers = [1, 2, 3, 4, 5]
result = list(filter(lambda x: x % 2 == 0, map(lambda x: x + 1, numbers)))

> 5번 문항

클로저(Closure)의 동작 방식에 대한 설명으로 가장 적절한 것은 무엇인가요?


A
함수 내부에서 전역 변수와 이름이 같은 변수를 선언하면 전역 변수 값이 즉시 덮어써지는 구조이다

B
외부 함수의 실행이 완전히 종료된 후에도 내부 함수가 외부 함수의 변수 값을 기억하고 유지하는 구조이다

C
여러 함수를 순차적으로 호출하여 결과를 하나의 리스트로 모아주는 구조이다

D
조건식의 참/거짓 결과에 따라 데이터를 걸러내는 구조이다

해설)
```text
클로저는 외부 함수 종료 후에도 내부 함수가 외부 함수의 환경(변수)을 캡슐화하여 계속 기억하는 구조입니다. 첫 번째 선지는 변수 은닉과 전역 변수 오염 방지 원리를 반대로 설명한 오답이며, 나머지는 각각 map()/filter()의 동작을 클로저로 오인한 매력적 오답입니다.
```

---
## 5. Key Takeaways


- closure 동작 방식
- memory 다시 복습
- map(), filter(), lambda
- decorator

---
## 6. Reflection

- closure 설명 들을 때 메모리에서 다시 헷갈렸다. 이미 다 이해했다고 생각했지만 막상 다른 개념이 들어와서 적용시켜 보니 내가 완전히 이해하진 않았구나를 깨달았다.
- 복습만이 아니라 이미 배웠던 개념을 다시 실습에서 적용시키는 과정이 제일 중요하다고 느꼈다. 하루에 몇가지 개념을 한꺼번에 배우다 보니, 다시 쓰지 않으면 금새 기억에서 휘발되어 사라져 버린다.
- 오늘 배운 이 많은 개념들은 실무에서, 또 프로젝트에서 어떻게 쓰일지 궁금하다. 
- 함수가 나올 때 수학을 포기했었는데, 지금 와서 다시 함수를 배우면서 재밌다고 느끼니 기분이 오묘하다. 
- 기억이 가물가물하지만, 몇 개월 전에 JS를 공부하면서 map을 배웠는데, 그때와 개념이 다른 것 같다. 몇몇 익숙한 개념들도 언어마다 조금씩 차이점이 있는 것 같다.
- 이제 2주차가 마무리되어 가는데, 총 40주차인 과정에서 38주가 남았다니 신기하기도 하고, 생각보다 시간이 많이 여유롭지는 않다고 느낀다. 평일은 빡세게 공부하고 주말은 여유롭게 쉬어갈까 생각도 했지만, 39주만 버티면 된다는 생각으로 주말에도 누구보다도 열심히 하려고 마음을 잡았다.
- 어제 봤던 테스트에서 생각보다 좋은 성적이 나오지 않았다. 풀 때는 한 문제 빼고 다 수월하게 풀었다고 자신했는데, 막상 점수는 기대치보다 매우 낮았다. 실습문제 많이 풀면서 반성해야겠다...